# CelebAMask-HQ CPU BG / HAIR / FACE 32-Sample Eval Package

Use this notebook with a CPU runtime. It creates a 32-sample held-out test eval
package from the already processed 5k `BG / HAIR / FACE` dataset.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/CelebMaskHQ_Colab')
REPO_DRIVE = DRIVE_ROOT / 'repo'
PROCESSED_DRIVE_ROOT = DRIVE_ROOT / 'processed'
PROJECT_LOCAL = Path('/content/project')

PROCESSED_NAME = 'processed_celebmaskhq_bghairface_5k'
PROCESSED_DRIVE = PROCESSED_DRIVE_ROOT / PROCESSED_NAME

EVAL32_PACKAGE_NAME = 'processed_celebmaskhq_bghairface_5k_test_eval_32'
EVAL32_OUTPUT_NAME = 'processed_celebmaskhq_bghairface_5k_test_eval_32_package'
EVAL32_OUTPUT_DRIVE = PROCESSED_DRIVE_ROOT / EVAL32_OUTPUT_NAME

MAX_EVAL_SAMPLES = 32
FORCE_REBUILD_EVAL32_PACKAGE = False

print('Repo on Drive:', REPO_DRIVE)
print('Processed dataset:', PROCESSED_DRIVE)
print('32-sample eval package:', EVAL32_OUTPUT_DRIVE)


In [ ]:
import shutil

assert REPO_DRIVE.exists(), f'Missing repo folder on Drive: {REPO_DRIVE}'
assert PROCESSED_DRIVE.exists(), f'Missing processed BG/HAIR/FACE 5k dataset: {PROCESSED_DRIVE}'
assert (PROCESSED_DRIVE / 'metadata' / 'layered_samples.jsonl').exists(), 'Missing processed layered metadata.'

if PROJECT_LOCAL.exists():
    shutil.rmtree(PROJECT_LOCAL)
shutil.copytree(REPO_DRIVE, PROJECT_LOCAL)
print('Copied repo to', PROJECT_LOCAL)


In [ ]:
import json

mapping = json.loads((PROCESSED_DRIVE / 'metadata' / 'mapping.json').read_text(encoding='utf-8'))
rows = [
    json.loads(line)
    for line in (PROCESSED_DRIVE / 'metadata' / 'layered_samples.jsonl').read_text(encoding='utf-8').splitlines()
    if line.strip()
]
test_rows = [row for row in rows if row.get('split') == 'test']

assert mapping['layered_export']['scheme'] == 'bg_hair_face'
assert mapping['layered_export']['target_layer_names'] == ['BG', 'HAIR', 'FACE']
assert len(test_rows) >= MAX_EVAL_SAMPLES, f'Need at least {MAX_EVAL_SAMPLES} test rows; found {len(test_rows)}'
assert all(row['layer_names'] == ['BG', 'HAIR', 'FACE'] for row in test_rows), test_rows[0]
assert all(row['layer_count'] == 3 for row in test_rows), test_rows[0]

print('Processed dataset verification passed.')
print('Available test rows:', len(test_rows))


In [ ]:
import json
import shutil
import subprocess
import sys

if FORCE_REBUILD_EVAL32_PACKAGE and EVAL32_OUTPUT_DRIVE.exists():
    shutil.rmtree(EVAL32_OUTPUT_DRIVE)

manifest_path = EVAL32_OUTPUT_DRIVE / 'package_manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    assert manifest['target_split'] == 'test', manifest['target_split']
    assert manifest['sample_count'] == MAX_EVAL_SAMPLES, manifest['sample_count']
    assert manifest['package_name'] == EVAL32_PACKAGE_NAME, manifest['package_name']
    print('32-sample eval package already exists and matches this notebook; skipping rebuild.')
else:
    command = [
        sys.executable,
        '-u',
        'scripts/package_qwen_layered_eval_subset.py',
        '--processed-root',
        str(PROCESSED_DRIVE),
        '--output-root',
        str(EVAL32_OUTPUT_DRIVE),
        '--package-name',
        EVAL32_PACKAGE_NAME,
        '--split',
        'test',
        '--max-samples',
        str(MAX_EVAL_SAMPLES),
        '--sample-strategy',
        'spaced',
        '--progress-every',
        '1',
    ]
    print('Running command:', ' '.join(command))
    subprocess.run(command, cwd=PROJECT_LOCAL, check=True)

assert manifest_path.exists(), f'Missing eval package manifest: {manifest_path}'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
print('32-sample eval package ready at:', EVAL32_OUTPUT_DRIVE)
print('Selected test sample ids:', manifest['selected_sample_ids'])


## Result

The 32-sample eval package is ready on Drive:

- `processed/processed_celebmaskhq_bghairface_5k_test_eval_32_package`
